# Heart Dataset: Descriptive Analysis

This notebook explores the supplied binary `Heart_Risk` label and evaluates two classifiers as an exercise in reproducing that label. The dataset files do not document the label-generation process or provenance. Results are not clinical evidence and must not be used for diagnosis or treatment.

## 1. Load and validate the supplied data

In [ ]:
import pandas as pd
from heart_analysis import load_dataset, data_quality_report

df = load_dataset()
quality = data_quality_report(df)
print('Shape:', df.shape)
print('Quality report:', quality)
df.head()

The file has 70,000 records and 19 fields. The recorded target has two equally sized values. Exact duplicate rows are counted and reported; the source file is never changed. The model evaluation excludes exact copies in memory before creating its holdout split, while descriptive summaries below use all supplied rows.

## 2. Exploratory data analysis

In [ ]:
import matplotlib.pyplot as plt

df['Heart_Risk'].value_counts().sort_index().plot(kind='bar', color=['#4a9fd8', '#e2725b'])
plt.xlabel('Supplied Heart_Risk label')
plt.ylabel('Records')
plt.title('Distribution of supplied labels')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
df.boxplot(column='Age', by='Heart_Risk', grid=False)
plt.title('Recorded age by supplied label')
plt.suptitle('')
plt.xlabel('Supplied Heart_Risk label')
plt.ylabel('Recorded age')
plt.tight_layout()
plt.show()

In [ ]:
df.describe().T

In [ ]:
df['Heart_Risk'].value_counts().sort_index().rename('record_count').to_frame()

In [ ]:
age_summary = df.assign(age_band=pd.cut(df['Age'], bins=[17, 29, 39, 49, 59, 69, 79, 90], include_lowest=True))
age_summary.groupby('age_band', observed=False)['Heart_Risk'].agg(['count', 'mean'])

In [ ]:
df.corr(numeric_only=True)['Heart_Risk'].sort_values(ascending=False).to_frame('correlation_with_supplied_label')

These tables show associations with the label as recorded. They do not demonstrate causation, a clinically meaningful target, or performance on an independently collected population. In the supplied file, label values are strongly associated with age and several binary fields, but the project files do not establish how those labels were assigned.

## 3. Holdout model evaluation

For predictable local runtime, the model evaluation uses a reproducible stratified sample of up to 20,000 unique rows, followed by a stratified 80/20 holdout. The descriptive tables above still use all supplied rows.

In [ ]:
from heart_analysis import train_models

evaluation = train_models(df, max_rows=20_000)
print('Unique complete rows available:', evaluation['model_rows'])
print('Stratified evaluation sample rows:', evaluation['sample_rows'])
print('Exact duplicates excluded from evaluation:', evaluation['rows_removed_as_exact_duplicates_for_evaluation'])
for model_name, result in evaluation['results'].items():
    print(f'\n{model_name}')
    print(pd.Series(result['metrics']).round(4))
    print('Per-class metrics:')
    print(pd.DataFrame.from_dict(result['class_metrics'], orient='index').round(4))
    print('Confusion matrix [label 0, label 1]:')
    print(result['confusion_matrix'])

In [ ]:
evaluation['results']['Random Forest']['feature_importances'].head(10).to_frame()

The split is stratified and uses a fixed random seed. Exact duplicate records are removed only from the in-memory modeling frame so copies do not appear across train and test. The Linear SVM scaler is fitted inside a pipeline on training data only. Random Forest does not require feature scaling. No PCA is applied because it was not used by the original final estimators.

The previous notebook fit a separate scaler on the test set and fit PCA separately on train and test; its saved metrics and PCA outputs are therefore superseded. The scores produced by this corrected run still evaluate agreement with the undocumented labels only.

## 4. Limitations

Dataset provenance, sampling method, label definition, and independent ground truth are not documented in the repository. This analysis is exploratory and is not a validated clinical prediction model.